<a href="https://colab.research.google.com/github/Adiba101/medical_chatbot_finetuning/blob/main/medical_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
print("AI with Adiba")

AI with Adiba


In [ ]:
#Step 1: Create & Setup hugging face API Token in collab

In [ ]:
# Step 2 : Install Required Dependencies

# Step A: Install the correct PyTorch version
!pip install torch==2.1.2 torchvision==0.16.1 torchaudio==2.1.2 --index-url https://download.pytorch.org/whl/cu121
# Step B: Install Unsloth (dev version)
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# Step C: Install other required libs
!pip install trl==0.8.6 peft==0.7.2 xformers==0.0.22.post7


Looking in indexes: https://download.pytorch.org/whl/cu121
ERROR: Could not find a version that satisfies the requirement torch==2.1.2 (from versions: 2.2.0+cu121, 2.2.1+cu121, 2.2.2+cu121, 2.3.0+cu121, 2.3.1+cu121, 2.4.0+cu121, 2.4.1+cu121, 2.5.0+cu121, 2.5.1+cu121)
ERROR: No matching distribution found for torch==2.1.2
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-ysp1u9hq/unsloth_b551594f01aa4ea38a56fefcf060a305
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-ysp1u9hq/unsloth_b551594f01aa4ea38a56fefcf060a305
  Resolved https://github.com/unslothai/unsloth.git to commit bda9e3d39b425f902d29e80c1f2870be7048d9c3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 18.4 MB/s eta 0:0

In [ ]:
#Step 3: Import necessary libraries
import unsloth
from unsloth import FastLanguageModel, is_bfloat16_supported
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import load_dataset
from huggingface_hub import login
import wandb


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [ ]:
#Step 4: check MED TOKEN
from google.colab  import userdata
med_token=userdata.get('MED_TOKEN')
login (med_token)

In [ ]:
#Optional : check GPU availability
#Test if CUDA is available
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

CUDA available: True
GPU device: Tesla T4


In [ ]:
#Step 5: Setup pretrained Deepseek-R1
model_name = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
max_sequence_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
model_name = model_name,
max_seq_length = max_sequence_length,
dtype = dtype,
load_in_4bit = load_in_4bit,
token = med_token
)



==((====))==  Unsloth 2025.11.4: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [ ]:
#Step 6 : setting system prompt
prompt_style = """
Below is a task description along with additional context provided in the input section. Your goal is to provide a well-reasoned response effectively addresses the requests.

Before crafting your answer , take a moment to carefully analyze the question . Develop a clear , step-by-step thought process to ensure your response is both logical and accurate.

### Task:
You are a medical expert specializing in clinical reasoning , diagonostics , and treatment planning . Answer the medical question below using your advanced knowledge.

### Query :
{}

### Answer:
<think>{}
"""

In [ ]:
# Step 7: Run Inference on the model

# Define a test question
question = """ A 61-year-old woman with a long historyy of involuntary urine loss during activities like oughing or
               sneezing but no leakage at night undergoes a gynecological exam and Q-tip test. Based on these fndings,
               what would cystometry most likely reveal about her residual volume and detrusor constractions?"""

FastLanguageModel.for_inference(model)

#Tokenize the input
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

#Generate a response
outputs = model.generate(
    input_ids = inputs.input_ids,
    attention_mask = inputs.attention_mask,
    max_new_tokens = 1200 ,
    use_cache = True
)

#Decode the response tokens back to test
response = tokenizer.batch_decode(outputs)
print(response)


["<｜begin▁of▁sentence｜>\nBelow is a task description along with additional context provided in the input section. Your goal is to provide a well-reasoned response effectively addresses the requests.\n\nBefore crafting your answer , take a moment to carefully analyze the question . Develop a clear , step-by-step thought process to ensure your response is both logical and accurate.\n\n### Task:\nYou are a medical expert specializing in clinical reasoning , diagonostics , and treatment planning . Answer the medical question below using your advanced knowledge.\n\n### Query :\n A 61-year-old woman with a long historyy of involuntary urine loss during activities like oughing or\n               sneezing but no leakage at night undergoes a gynecological exam and Q-tip test. Based on these fndings,\n               what would cystometry most likely reveal about her residual volume and detrusor constractions?\n\n### Answer:\n<think>\nOkay, so I'm trying to figure out what the cystometry would sh

In [ ]:
print (response[0].split("### Answer:")[1])


<think>
Okay, so I'm trying to figure out what the cystometry would show for this 61-year-old woman. Let me break it down step by step.

First, the patient has a history of involuntary urine loss when she coughs or sneezes. That makes me think of stress urinary incontinence. I remember that stress incontinence is usually due to the urethral sphincter not closing properly during activities like coughing. So, the issue isn't with the detrusor muscle, which is responsible for the actual storage and emptying of urine, but with the sphincter's ability to seal the urethra.

Now, she underwent a gynecological exam and a Q-tip test. I'm a bit fuzzy on what exactly the Q-tip test entails. From what I recall, the Q-tip is a small catheter that measures the pressure in the bladder. It's used to determine if there's an overactive bladder or if the detrusor muscle is hyperactive. If the detrusor is hyperactive, it might lead to urge incontinence, where the person has a strong, sudden need to urina

In [ ]:
# Step 8 : Setup fine-tuning

#Load Dataset
medical_dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT","en",split="train[:500]")



README.md: 0.00B [00:00, ?B/s]

medical_o1_sft.json:   0%|          | 0.00/58.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/19704 [00:00<?, ? examples/s]

In [ ]:
medical_dataset[1]

{'Question': 'A 33-year-old woman is brought to the emergency department 15 minutes after being stabbed in the chest with a screwdriver. Given her vital signs of pulse 110/min, respirations 22/min, and blood pressure 90/65 mm Hg, along with the presence of a 5-cm deep stab wound at the upper border of the 8th rib in the left midaxillary line, which anatomical structure in her chest is most likely to be injured?',
 'Complex_CoT': "Okay, let's figure out what's going on here. A woman comes in with a stab wound from a screwdriver. It's in her chest, upper border of the 8th rib, left side, kind of around the midaxillary line. First thought, that's pretty close to where the lung sits, right?\n\nLet's talk about location first. This spot is along the left side of her body. Above the 8th rib, like that, is where a lot of important stuff lives, like the bottom part of the left lung, possibly the diaphragm too, especially considering how deep the screwdriver went.\n\nThe wound is 5 cm deep. Tha

In [ ]:
EOS_TOKEN = tokenizer.eos_token # Define EOS_TOKEN which tells the model when to stop generating text during training
EOS_TOKEN

'<｜end▁of▁sentence｜>'

In [ ]:
# Update the prompt_style for training
train_prompt_style = """
Below is a task description along with additional context provided in the input section. Your goal is to provide a well-reasoned response effectively addresses the requests.

Before crafting your answer , take a moment to carefully analyze the question . Develop a clear , step-by-step thought process to ensure your response is both logical and accurate.

### Instructions:
You are a medical expert specializing in clinical reasoning , diagonostics , and treatment planning . Answer the medical question below using your advanced knowledge.

### Question :
{}

### Response:
<think>
{}
</think>
{}
"""

In [ ]:
# Prepare data for fine-tuning

def preprocess_input_data(examples):
  inputs = examples["Question"]
  cots = examples["Complex_CoT"]
  outputs = examples["Response"]

  texts = []

  for input, cot, output in zip(inputs, cots, outputs):
    text = train_prompt_style.format(input, cot , output) + EOS_TOKEN
    texts.append(text)

  return {
      "texts" : texts,
  }

In [ ]:
finetune_dataset = medical_dataset.map(preprocess_input_data, batched = True)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
finetune_dataset["texts"][0]

"\nBelow is a task description along with additional context provided in the input section. Your goal is to provide a well-reasoned response effectively addresses the requests.\n\nBefore crafting your answer , take a moment to carefully analyze the question . Develop a clear , step-by-step thought process to ensure your response is both logical and accurate.\n\n### Instructions:\nYou are a medical expert specializing in clinical reasoning , diagonostics , and treatment planning . Answer the medical question below using your advanced knowledge.\n\n### Question :\nGiven the symptoms of sudden weakness in the left arm and leg, recent long-distance travel, and the presence of swollen and tender right lower leg, what specific cardiac abnormality is most likely to be found upon further evaluation that could explain these findings?\n\n### Response:\n<think>\nOkay, let's see what's going on here. We've got sudden weakness in the person's left arm and leg - and that screams something neuro-rela

In [ ]:
# Step 9 : Setup/Apply LoRA finetuning to the model

model_lora = FastLanguageModel.get_peft_model(
    model = model,
    r = 16,
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3047,
    use_rslora = False ,
    loftq_config = None

  )

Unsloth 2025.11.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
#Add this before creating the trainer
if hasattr(model, '_unwrapped_old_generate'):
  del model._unwrapped_old_generate

In [ ]:
# ------------------------------
# 1. BUILD RAW DATA LIST HERE
# ------------------------------
# ❗❗ Replace this with your actual dataset list
raw_list = [
    {"texts": "Below is a task description ... <｜end▁of▁sentence｜>"},
    # Add ALL your 500 examples here
]

# ------------------------------
# 2. VALIDATE RAW DATA
# ------------------------------
if len(raw_list) == 0:
    raise ValueError("❌ ERROR: raw_list is EMPTY. Add your dataset to raw_list first!")

for i, item in enumerate(raw_list[:3]):
    if "texts" not in item:
        raise KeyError(f"❌ Example {i} has no 'texts' key: {item}")

print("✅ Raw list length:", len(raw_list))


# ------------------------------
# 3. BUILD CLEAN HF DATASET
# ------------------------------
from datasets import Dataset
finetune_dataset = Dataset.from_list(raw_list)

print("Dataset size:", len(finetune_dataset))
print("Keys:", finetune_dataset.column_names)


# ------------------------------
# 4. formatting_func MUST return list[str]
# ------------------------------
def formatting_func(example):
    return [str(example["texts"]).strip()]


# ------------------------------
# 5. Create Trainer
# ------------------------------
trainer = SFTTrainer(
    model=model_lora,
    tokenizer=tokenizer,
    train_dataset=finetune_dataset,
    dataset_text_field="texts",
    formatting_func=formatting_func,
    dataset_num_proc=1,
    max_seq_length=max_sequence_length,
    packing=False,

    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=1,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
    )
)

print("Trainer created successfully!")

# ------------------------------
# 6. Train
# ------------------------------
trainer_stats = trainer.train()
print("Training Done!")


✅ Raw list length: 1
Dataset size: 1
Keys: ['texts']


num_proc must be <= 1. Reducing num_proc to 1 for dataset of size 1.


Unsloth: Tokenizing ["text"] (num_proc=1):   0%|          | 0/1 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


Trainer created successfully!


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1 | Num Epochs = 60 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:



KeyboardInterrupt: 

In [ ]:
finetune_dataset.column_names
len(finetune_dataset)
finetune_dataset[0]


In [ ]:
# Setup WANDB
from google.colab import userdata
import wandb

#login secret
wnb_token = userdata.get("WANDB_API_TOKEN")

#Login to WnB
wandb.login (key=wnb_token)  # import wandb
print("Token length:", len(wnb_token))   # Check

run = wandb.init(
    project = 'Fine-tune-DeepSeek=R1-on-Medical-CoT-Dataset',
    job_type = "training" ,
    anonymous ="allow"
)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: adiba13ansari (adiba13ansari-doon-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Token length: 40


wandb: Detected [huggingface_hub.inference, openai] in use.
wandb: Use W&B Weave for improved LLM call tracing. Install Weave with `pip install weave` then add `import weave` to the top of your script.
wandb: For more information, check out the docs at: https://weave-docs.wandb.ai/


In [ ]:
# Start fine-tuning process
trainer_stats = trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1 | Num Epochs = 60 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
10,6.389200
20,0.786700
30,0.009300
40,0.000000
50,0.000000
60,0.000000


train/epoch,▁▂▄▅▇██
train/global_step,▁▂▄▅▇██
train/grad_norm,▃█▁▁▁▁
train/learning_rate,█▇▅▄▂▁
train/loss,█▂▁▁▁▁
total_flos,29885596139520.0
train/epoch,60
train/global_step,60
train/grad_norm,0.00256
train/learning_rate,1e-05
train/loss,0


In [ ]:
wandb.finish()

In [ ]:
# Step 10 : Testing after fine - tuning
question = """ A 61-year-old woman with a long historyy of involuntary urine loss during activities like oughing or
               sneezing but no leakage at night undergoes a gynecological exam and Q-tip test. Based on these fndings,
               what would cystometry most likely reveal about her residual volume and detrusor constractions?"""

FastLanguageModel.for_inference(model_lora)

#Tokenize the input
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

#Generate a response
outputs = model_lora.generate(
    input_ids = inputs.input_ids,
    attention_mask = inputs.attention_mask,
    max_new_tokens = 1200 ,
    use_cache = True
)

#Decode the response tokens back to test
response = tokenizer.batch_decode(outputs)
print(response)


["<｜begin▁of▁sentence｜>\nBelow is a task description along with additional context provided in the input section. Your goal is to provide a well-reasoned response effectively addresses the requests.\n\nBefore crafting your answer , take a moment to carefully analyze the question . Develop a clear , step-by-step thought process to ensure your response is both logical and accurate.\n\n### Task:\nYou are a medical expert specializing in clinical reasoning , diagonostics , and treatment planning . Answer the medical question below using your advanced knowledge.\n\n### Query :\n A 61-year-old woman with a long historyy of involuntary urine loss during activities like oughing or\n               sneezing but no leakage at night undergoes a gynecological exam and Q-tip test. Based on these fndings,\n               what would cystometry most likely reveal about her residual volume and detrusor constractions?\n\n### Answer:\n<think>\nOkay, so I'm trying to figure out what the cystometry would sh

In [ ]:
print(response[0].split("### Answer:")[1])


<think>
Okay, so I'm trying to figure out what the cystometry would show for this 61-year-old woman. Let's break it down step by step. 

First, the patient has a history of involuntary urine loss during activities like coughing or sneezing. That makes me think of stress urinary incontinence, especially since she doesn't leak at night. Stress incontinence is typically due to <｜end▁of▁sentence｜>


In [ ]:
question = """Considering a patient with a rare genetic disorder sickle cell disease,
 who is now showing new neurological symptoms, describe how the underlying condition might influence the
 presentation and potential treatment options for the new symptoms?"""
FastLanguageModel.for_inference(model_lora)

#Tokenize the input
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

#Generate a response
outputs = model_lora.generate(
    input_ids = inputs.input_ids,
    attention_mask = inputs.attention_mask,
    max_new_tokens = 1200 ,
    use_cache = True
)

#Decode the response tokens back to test
response = tokenizer.batch_decode(outputs)

print(response[0].split("### Answer:")[1])



<think>
Okay, so I'm trying to figure out how sickle cell disease (SCD) might present new neurological symptoms and the possible treatment options. Let me start by recalling what I know about SCD. SCD is a rare genetic disorder where the hemoglobin (the protein in red blood cells that carries oxygen) is abnormal. This abnormal hemoglobin can cause red blood cells to break down prematurely, leading to various health issues like pain crises, organ damage, and an increased risk of infections.

Now, the patient in question has SCD and is showing new neurological symptoms. I need to think about how SCD could be causing these symptoms. I know that SCD can affect various organ systems, including the brain. I've heard that people with SCD can have issues like stroke, which is a neurological event. So, maybe the new symptoms are related to a stroke or another cerebrovascular issue.

Let me break this down. First, how does SCD contribute to neurological symptoms? Well, SCD can cause vaso-occlus